In [ ]:
# @title Script
if 'read_token' not in globals():
    from huggingface_hub import login
    Custom_1 = "hf_WorGiwfbyFCqKT"
    Custom_2 = "fwwKjTXYBhDjLWFgtYKE"
    Custom_Hg_Token = f"{Custom_1}{Custom_2}"
    read_token = Custom_Hg_Token
    try:
        login(token=read_token)
    except Exception as e:
        print(f"⚠️ Không đăng nhập được Hugging Face, model bị khoá (gated) sẽ không tải được: {e}")

import os, gc, io, base64, json, yaml
import subprocess, threading, socket, time
import torch
from pathlib import Path
from PIL import Image
from unittest.mock import patch
from IPython.display import clear_output

if torch.cuda.is_available():
  gpu_name = torch.cuda.get_device_name()
  gpu_capability = torch.cuda.get_device_capability()
else:
  gpu_name, gpu_capability = "CPU", (0, 0)
  print("⚠️ Runtime chưa có GPU: vào Runtime > Change runtime type để chọn GPU trước khi train")
print(gpu_name)
# G4 = NVIDIA RTX PRO 6000 Blackwell Server Edition (sm_120)
is_g4 = "RTX PRO 6000" in gpu_name or gpu_capability[0] == 12
if 'A100' in gpu_name:
  os.environ['TORCH_CUDA_ARCH_LIST'] = '8.0'
if 'L4' in gpu_name:
  os.environ['TORCH_CUDA_ARCH_LIST'] = '8.9'
if 'T4' in gpu_name:
  os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5'
if is_g4:
  os.environ['TORCH_CUDA_ARCH_LIST'] = '12.0'

def delete_specific_models():
    variable_names = ['clip_model', 'tokenizer', 'text_model', 'image_adapter', 'model', 'processor']
    for var_name in variable_names:
        if var_name in globals():
            del globals()[var_name]
    gc.collect()
    torch.cuda.empty_cache()

model_list = {
    "APIGemini | 3.8 Flash": "gemini-3.8-flash",
    "APIGemini | 3.5 Flash Lite": "gemini-3.5-flash-lite",
}

def encode_image(image):
    with io.BytesIO() as image_buffer:
        image.save(image_buffer, format="PNG")
        image_buffer.seek(0)
        encoded_image = base64.b64encode(image_buffer.read()).decode('utf-8')
    return encoded_image

def api_check():
    api_file = os.path.join(data_dir,"Setting/API_key_for_sdvn_comfy_node.json")
    if os.path.exists(api_file):
        with open(api_file, 'r', encoding='utf-8') as f:
            api_list = json.load(f)
        return api_list
    else:
        return None

def api_caption(image, length:int, APIkey, Caption, prompt):
    Caption = Caption.strip()
    if Caption not in model_list:
        raise ValueError(f"Caption '{Caption}' không có trong model_list: {list(model_list)}")
    if APIkey == "":
        api_list = api_check()
        if api_list != None:
            if "Gemini" in Caption:
                APIkey =  api_list["Gemini"]
            if "OpenAI" in Caption:
                APIkey =  api_list["OpenAI"]
    model_name = model_list[Caption]
    prompt += f"Picture description, Send the description on demand, limit {length} words, only send me the answer, Always return English. "
    answer = ""
    if 'Gemini' in Caption:
        from google import genai
        client = genai.Client(api_key=APIkey)
        response = client.models.generate_content(
                    model=model_name,
                    contents=[prompt, image])
        answer = response.text
    if "OpenAI" in Caption:
        from openai import OpenAI
        client = OpenAI(api_key=APIkey)
        image = encode_image(image)
        prompt = [{"type": "input_text", "text": prompt, },
                  {"type": "input_image", "image_url": f"data:image/jpeg;base64,{image}"},]
        messages = [{"role": "user", "content": prompt}]
        response = client.responses.create(
            model=model_name,
            input=messages
        )
        answer = response.output_text
    return answer.strip()

def clean_directory(directory):
  # Giữ ảnh (không phân biệt hoa thường), caption .txt và cache .safetensors; xoá các file khác
  keep_types = (".png", ".jpg", ".jpeg", ".webp", ".bmp", ".txt", ".safetensors") + video_extensions
  for item in os.listdir(directory):
      file_path = os.path.join(directory, item)
      if os.path.isfile(file_path):
          file_ext = os.path.splitext(item)[1].lower()
          if file_ext not in keep_types:
              print(f"Deleting file {item} from {directory}")
              os.remove(file_path)
      elif os.path.isdir(file_path):
          clean_directory(file_path)

#Florence

version = "large"
device = "cuda:0" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    torch_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    torch_dtype = torch.float32
florence_native = True

def load_model(version, device):
    # ai-toolkit ghim transformers 5.x, code remote cũ của microsoft/Florence-2 không còn chạy được.
    # Dùng Florence-2 có sẵn trong transformers (>= 4.56) với checkpoint florence-community,
    # chỉ quay về code remote khi transformers quá cũ (VD: môi trường Caption_Tools).
    global florence_native
    from transformers import AutoProcessor
    model_dir = "/content/Model"
    os.makedirs(model_dir, exist_ok=True)

    try:
        from transformers import Florence2ForConditionalGeneration
    except ImportError:
        Florence2ForConditionalGeneration = None

    if Florence2ForConditionalGeneration is not None:
        florence_native = True
        identifier = "florence-community/Florence-2-" + version
        model = Florence2ForConditionalGeneration.from_pretrained(identifier, cache_dir=model_dir)
        processor = AutoProcessor.from_pretrained(identifier, cache_dir=model_dir)
    else:
        florence_native = False
        from transformers import AutoModelForCausalLM
        from transformers.dynamic_module_utils import get_imports

        def fixed_get_imports(filename):
            """Workaround for FlashAttention"""
            imports = get_imports(filename)
            if os.path.basename(filename) == "modeling_florence2.py" and "flash_attn" in imports:
                imports.remove("flash_attn")
            return imports

        identifier = "microsoft/Florence-2-" + version
        with patch("transformers.dynamic_module_utils.get_imports", fixed_get_imports):
            model = AutoModelForCausalLM.from_pretrained(identifier, torch_dtype=torch_dtype, cache_dir=model_dir, trust_remote_code=True, attn_implementation="eager")
            processor = AutoProcessor.from_pretrained(identifier, cache_dir=model_dir, trust_remote_code=True)

    model = model.to(device, dtype=torch_dtype).eval()

    return (model, processor)

def load(version, device):
  if 'processor' not in globals():
    global model, processor
    model, processor = load_model(version, device)

def florence_caption(task_prompt="caption", image=None, max_new_tokens = 1024, num_beams = 3, do_sample = False, fill_mask = False, text_input=None):
    """Generate Florence-2 captions using the prompt tokens defined in ComfyUI-Florence2/nodes.py.
    The fill_mask argument is kept for backwards compatibility.
    """
    prompt_tokens = {
        "region_caption": "<OD>",
        "dense_region_caption": "<DENSE_REGION_CAPTION>",
        "region_proposal": "<REGION_PROPOSAL>",
        "caption": "<CAPTION>",
        "detailed_caption": "<DETAILED_CAPTION>",
        "more_detailed_caption": "<MORE_DETAILED_CAPTION>",
        "caption_to_phrase_grounding": "<CAPTION_TO_PHRASE_GROUNDING>",
        "referring_expression_segmentation": "<REFERRING_EXPRESSION_SEGMENTATION>",
        "ocr": "<OCR>",
        "ocr_with_region": "<OCR_WITH_REGION>",
        "docvqa": "<DocVQA>",
        "prompt_gen_tags": "<GENERATE_TAGS>",
        "prompt_gen_mixed_caption": "<MIXED_CAPTION>",
        "prompt_gen_analyze": "<ANALYZE>",
        "prompt_gen_mixed_caption_plus": "<MIXED_CAPTION_PLUS>",
    }
    token_input = "caption" if task_prompt is None else str(task_prompt)
    normalized_task = token_input.lower()
    if token_input.startswith("<") and token_input.endswith(">"):
        task_token = token_input
    else:
        task_token = prompt_tokens.get(normalized_task, token_input)

    extra_input = ""
    if text_input:
        extra_input = str(text_input).strip()

    text_allowed = {
        "referring_expression_segmentation",
        "caption_to_phrase_grounding",
        "docvqa",
    }
    if extra_input and normalized_task in prompt_tokens and normalized_task not in text_allowed:
        raise ValueError("Text input is only supported for referring_expression_segmentation, caption_to_phrase_grounding, and docvqa.")

    if normalized_task == "docvqa" and not extra_input:
        raise ValueError("DocVQA requires a text_input prompt.")

    prompt = task_token if not extra_input else f"{task_token} {extra_input}"

    with torch.no_grad():
        inputs = processor(text=prompt, images=image, return_tensors="pt").to(device, torch_dtype)
        generate_kwargs = dict(
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            num_beams=num_beams,
        )
        if not florence_native:
            # code remote cũ lỗi cache với transformers mới
            generate_kwargs["use_cache"] = False
        generated_ids = model.generate(**inputs, **generate_kwargs)

    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
    try:
        parsed = processor.post_process_generation(generated_text, task=task_token, image_size=(image.width, image.height))
        answer = parsed.get(task_token) if isinstance(parsed, dict) else parsed
        if isinstance(answer, str):
            return answer.strip()
    except Exception as e:
        print(f"Florence post-process lỗi, dùng text thô: {e}")
    cleaned = generated_text.replace("</s>", "").replace("<s>", "").replace("<pad>", "").strip()

    return cleaned

#Florence

def caption_dir(image_dir,prompt):
  if Caption == 'Florence':
    load(version, device)
  for img_file in os.listdir(image_dir):
      file_path = os.path.join(image_dir, img_file)
      if os.path.isdir(file_path) :
          caption_dir(file_path,prompt)
      if img_file.lower().endswith((".png", ".jpg", ".jpeg", ".webp", ".bmp", ".PNG", ".JPG", ".JPEG")):
          img_path = os.path.join(image_dir, img_file)
          image = Image.open(img_path).convert("RGB")
          if Caption == 'Florence':
            cap = florence_caption(prompt,image).replace('The image shows','')
          else:
            cap = api_caption(image, Cap_prompt[Caption_Length][3], APIkey, Caption, API_Prompt)
          txt_path = os.path.join(image_dir, f"{os.path.splitext(img_file)[0]}{extension}")
          with open(txt_path, "w") as f:
              f.write(cap)
          print(f"Miêu tả của ảnh {img_file}: {cap}")
  torch.cuda.empty_cache()
def read_file(filename):
    with open(filename, "r") as f:
        contents = f.read()
    return contents

def write_file(filename, contents):
    with open(filename, "w") as f:
        f.write(contents)

def process_tags(filename, custom_tag, append, remove_tag):
    contents = read_file(filename)
    if remove_tag:
      contents = contents.replace(custom_tag, "")
    else:
      tags = [tag.strip() for tag in contents.split(',')]
      custom_tags = [tag.strip() for tag in custom_tag.split(',')]
      for custom_tag in custom_tags:
          custom_tag = custom_tag.replace("_", " ")
          if custom_tag not in tags:
              if append:
                  tags.append(custom_tag)
              else:
                  tags.insert(0, custom_tag)
      contents = ', '.join(tags)
    write_file(filename, contents)

def check_dir(image_dir):
  if not any([filename.endswith(extension) for filename in os.listdir(image_dir)]):
      for filename in os.listdir(image_dir):
          if filename.lower().endswith((".png", ".jpg", ".jpeg", ".webp", ".bmp") + video_extensions):
              open(
                  os.path.join(image_dir, os.path.splitext(filename)[0] + extension),
                  "w",
              ).close()

def process_dir(image_dir, tag, append, remove_tag):
  check_dir(image_dir)
  for filename in os.listdir(image_dir):
      file_path = os.path.join(image_dir, filename)
      if os.path.isdir(file_path) :
          print(filename)
          process_dir(file_path, tag, append, remove_tag)
      elif filename.endswith(extension):
          process_tags(file_path, tag, append, remove_tag)

def add_forder_name(folder):
  for filename in os.listdir(folder):
    file_path = os.path.join(folder, filename)
    if os.path.isdir(file_path):
      folder_name = os.path.basename(file_path)
      try:
          steps, name = folder_name.split('_', 1)
          steps = int(steps)
      except ValueError:
          name = folder_name
      name = name.replace("/", ", ")
      process_dir(file_path, name, False, False)
      add_forder_name(file_path)

def get_steps(folder):
    folder_name = os.path.basename(folder)
    try:
        steps, name = folder_name.split('_', 1)
        steps = int(steps)
    except ValueError:
        steps = Steps
        name = folder_name
    return steps, name

def random_sample(folder, control_folder, fallback="girl portrait, smile"):
  import random
  file_list = get_supported_media(control_folder) or get_supported_media(folder)
  if not file_list:
    return [fallback, ""]
  control_path = random.choice(file_list)
  data_path = control_path.replace(control_folder, folder, 1)
  txt_path = os.path.splitext(data_path)[0] + ".txt"
  try:
    sample = read_file(txt_path).strip()
  except OSError:
    sample = ""
  if sample == "":
    sample = fallback
  sample = sample.replace('"', r'\"')
  return [sample, control_path]

video_extensions = (".mp4", ".avi", ".mov", ".webm", ".mkv", ".wmv", ".m4v", ".flv")
# Type train được từ video: ai-toolkit đọc cả video lẫn ảnh trong thư mục train
video_types = ["MiniMax-H3 (Video)"]

def get_supported_videos(folder):
  videos = []
  for root, dirs, files in os.walk(folder):
    dirs[:] = [d for d in dirs if not d.startswith('.')]
    videos += [os.path.join(root, f) for f in files if f.lower().endswith(video_extensions) and not f.startswith('.')]
  return videos

def get_supported_media(folder):
  return get_supported_images(folder) + get_supported_videos(folder)

def get_train_files(folder):
  # Ảnh, kèm video khi type đang chọn train được từ video
  if globals().get("TypeTrain") in video_types:
    return get_supported_media(folder)
  return get_supported_images(folder)

def get_supported_images(folder):
  import glob
  supported_extensions = (".png", ".jpg", ".jpeg", ".webp", ".bmp", ".PNG", ".JPG", ".JPEG")
  list_img = [file for ext in supported_extensions for file in glob.glob(f"{folder}/*{ext}")]
  for img_file in os.listdir(folder):
      file_path = os.path.join(folder, img_file)
      if os.path.isdir(file_path) :
          list_img = list_img + get_supported_images(file_path)
  return list_img

default_value = {
    "config_dir" : "/content/SDVN-training-colab-flux/Config",
    "Lora_name" : "lora_name",
    "folder_train": "/content/drive/MyDrive/SD-Data",
    "OutputFolder": "/content/drive/MyDrive/SD-Data/Lora",
    "Custom_Caption": "",
    "Dim": 32,
    "Alpha": 16,
    "Save_steps": 1000,
    "Resolution": "1024",
    "Batch_size": 4,
    "Train_TexEncoder": False,
    "Lr": 1e-4,
    "Sampler_Steps": 100,
    "Low_VRAM": False,
    "Sampler_Prompt": "",
    "TypeTrain": "Flux",
    "ControlFolder": "",
    "ControlFolder_2": "",
    "ControlFolder_3": "",
    "Control_Img_Path": "",
    "Lr_Scheduler": "constant",
    "Timestep_type": "shift",
    "Cache_latents": False,
    "Wandb_API": "",
    "Steps": 1000,
}
def check_value(default_value):
    # Gán giá trị mặc định cho tham số mà notebook đang chạy không khai báo
    for key, value in default_value.items():
        if key not in globals():
            globals()[key] = value

def check_folder_train(folder):
    images, videos = get_supported_images(folder), get_supported_videos(folder)
    if images or videos:
      print('=====================')
      print(f'Thư mục train: {folder}')
      print(f'  Số lượng ảnh: {len(images)}')
      if videos:
        print(f'  Số lượng video: {len(videos)} (chỉ dùng khi train {", ".join(video_types)})')
        caption_ext = globals().get('extension', '.txt')
        no_caption = [v for v in videos if not os.path.exists(os.path.splitext(v)[0] + caption_ext)]
        if no_caption:
          print(f'  ⚠️ {len(no_caption)} video chưa có caption {caption_ext} (Florence/Gemini chỉ caption ảnh)')
      print('=====================')
    else:
      print(f"Thư mục [ {folder} ] không có ảnh (.png, .jpg, .jpeg, .webp, .bmp) hoặc video (.mp4, .mov, .webm, ...) được hỗ trợ, hãy kiểm tra lại")

type_config = {
    "Qwen-Image-2.1": ["Comfy-Org/Qwen-Image-2.1","qwen_image_2"],
    "Flux": ["black-forest-labs/FLUX.1-dev","flux"],
    "Flux Kontext": ["black-forest-labs/FLUX.1-Kontext-dev","flux_kontext"],
    "Qwen": ["Qwen/Qwen-Image","qwen_image"],
    "Z-Image": ["Tongyi-MAI/Z-Image","zimage"],
    "Qwen-Edit": ["Qwen/Qwen-Image-Edit","qwen_image_edit"],
    "Hi-Dream": ["HiDream-ai/HiDream-I1-Full","hidream"],
    "Hi-Dream E1": ["HiDream-ai/HiDream-E1-1","hidream_e1"],
    "Wan2.2 TI2V 5B": ["Wan-AI/Wan2.2-TI2V-5B-Diffusers","wan22_5b"],
    "Wan2.1 I2V (14B-720P)": ["Wan-AI/Wan2.1-I2V-14B-720P-Diffusers", "wan21_i2v:14b"],
    "Wan2.1 I2V (14B-480P)": ["Wan-AI/Wan2.1-I2V-14B-480P-Diffusers","wan21_i2v:14b480p"],
    "Wan2.1 (14B)": ["Wan-AI/Wan2.1-T2V-14B-Diffusers","wan21:14b"],
    "Wan2.2 (14B)": ["ai-toolkit/Wan2.2-T2V-A14B-Diffusers-bf16","wan22_14b:t2v"],
    "Wan2.2 I2V (14B)": ["ai-toolkit/Wan2.2-I2V-A14B-Diffusers-bf16","wan22_14b_i2v"],
    "Z-Image Turbo": ["Tongyi-MAI/Z-Image-Turbo","zimage:turbo", "ostris/zimage_turbo_training_adapter/zimage_turbo_training_adapter_v2.safetensors"],
    "Z-Image De Turbo": ["ostris/Z-Image-De-Turbo","zimage:deturbo", "Tongyi-MAI/Z-Image-Turbo"],
    "Flux.2": ["black-forest-labs/FLUX.2-dev","flux2"],
    "FLUX.2-klein-base-9B": ["/content/flux-2-klein-base-9b.safetensors","flux2_klein_9b"],
    "FLUX.2-klein-base-4B": ["black-forest-labs/FLUX.2-klein-base-4B","flux2_klein_4b"],
    "Qwen-Edit-2509": ["Qwen/Qwen-Image-Edit-2509","qwen_image_edit_plus"],
    "Qwen-Edit-2511": ["Qwen/Qwen-Image-Edit-2511","qwen_image_edit_plus:2511"],
    "Qwen-Image-2512": ["Qwen/Qwen-Image-2512","qwen_image:2512"],
    "Krea2": ["/content/krea2_raw.safetensors","krea2"],
    "FLUX.3-image-klein-base": ["black-forest-labs/flux-3-image-klein-base","flux3"],
    "MiniMax-H3-Image": ["Comfy-Org/MiniMax-H3-Image","minimax_h3_image"],
    "MiniMax-H3 (Video)": ["Comfy-Org/MiniMax-H3","minimax_h3"],
}

# Type dùng nhiều ảnh control (control_path_1/2/3)
multi_control_types = ["Qwen-Image-2.1", "FLUX.2-klein-base-9B", "FLUX.2-klein-base-4B", "FLUX.3-image-klein-base", "MiniMax-H3-Image", "Qwen-Edit-2509", "Qwen-Edit-2511"]
# Type gắn --ctrl_img vào prompt sample khi có ControlFolder
ctrl_sample_types = ['Qwen-Image-2.1', 'Flux Kontext', "FLUX.2-klein-base-9B", "FLUX.2-klein-base-4B", "FLUX.3-image-klein-base", "MiniMax-H3-Image", 'Qwen-Edit', "Qwen-Edit-2509", "Qwen-Edit-2511", 'Hi-Dream E1', "Wan2.1 I2V (14B-480P)", "Wan2.1 I2V (14B-720P)", "Wan2.2 I2V (14B)"]
# Cạnh ngắn tối thiểu của ảnh sample khi lấy kích thước theo ảnh control
# (MiniMax-H3-Image được train cho ảnh 4MP trở lên, ảnh nhỏ hơn dễ lỗi)
sample_min_side = {"MiniMax-H3-Image": 2048}
# G4 (RTX PRO 6000 Blackwell): đổi qfloat8 -> convrot8. Khi train, convrot8 chạy forward bằng
# int8 tensor core, còn qfloat8 (quanto) phải giải nén cả trọng số sang bf16 ở mỗi bước.
G4_CONVROT8 = True
# Ghi đè thêm theo type sau khi đã lấy mặc định của ai-toolkit (đường dẫn tính từ process[0]),
# VD: {"Krea2": {"sample.sample_steps": 30}}
type_overrides = {}

#Toolkit defaults
# Đọc thông số mặc định theo từng model từ UI của ai-toolkit (extensions_built_in/*/ui.tsx),
# lấy giá trị "khi được chọn" của mỗi khoá "config.process[0]...." trong mục defaults.
import glob as _glob
import re as _re

_JS_UNKNOWN = object()

class _JsReader:
    def __init__(self, text, pos=0):
        self.t, self.i = text, pos

    def ws(self):
        t = self.t
        while self.i < len(t):
            if t[self.i].isspace():
                self.i += 1
            elif t.startswith("//", self.i):
                j = t.find("\n", self.i)
                self.i = len(t) if j < 0 else j + 1
            elif t.startswith("/*", self.i):
                j = t.find("*/", self.i)
                self.i = len(t) if j < 0 else j + 2
            else:
                break

    def peek(self):
        self.ws()
        return self.t[self.i] if self.i < len(self.t) else ""

    def expect(self, ch):
        if self.peek() != ch:
            raise ValueError(f"expected {ch!r} at {self.i}")
        self.i += 1

    def string(self):
        q = self.t[self.i]
        self.i += 1
        out = []
        while self.t[self.i] != q:
            c = self.t[self.i]
            if c == "\\":
                self.i += 1
                c = {"n": "\n", "t": "\t"}.get(self.t[self.i], self.t[self.i])
            elif q == "`" and self.t.startswith("${", self.i):
                raise ValueError("template literal")
            out.append(c)
            self.i += 1
        self.i += 1
        return "".join(out)

    def skip_expression(self):
        # bỏ qua một biểu thức không phải literal (biến, hàm, JSX...) tới dấu , ] } ở cùng cấp
        depth = 0
        while self.i < len(self.t):
            c = self.t[self.i]
            if c in "\"'`":
                self.string()
                continue
            if c in "([{":
                depth += 1
            elif c in ")]}":
                if depth == 0:
                    return
                depth -= 1
            elif c == "," and depth == 0:
                return
            self.i += 1

    def value(self):
        c = self.peek()
        if c == "[":
            self.i += 1
            items = []
            while self.peek() != "]":
                items.append(self.value())
                if self.peek() == ",":
                    self.i += 1
            self.i += 1
            return _JS_UNKNOWN if any(v is _JS_UNKNOWN for v in items) else items
        if c == "{":
            self.i += 1
            obj, unknown = {}, False
            while self.peek() != "}":
                if self.t.startswith("...", self.i):
                    self.skip_expression()
                    unknown = True
                else:
                    key = self.string() if self.peek() in "\"'" else self.ident()
                    self.expect(":")
                    obj[key] = self.value()
                    unknown = unknown or obj[key] is _JS_UNKNOWN
                if self.peek() == ",":
                    self.i += 1
            self.i += 1
            return _JS_UNKNOWN if unknown else obj
        if c in "\"'`":
            try:
                return self.string()
            except ValueError:
                self.skip_expression()
                return _JS_UNKNOWN
        m = _re.compile(r"-?(\d+\.?\d*|\.\d+)([eE][-+]?\d+)?").match(self.t, self.i)
        if m and self.t[self.i] in "-.0123456789":
            self.i = m.end()
            num = m.group(0)
            return float(num) if any(ch in num for ch in ".eE") else int(num)
        word = _re.compile(r"[A-Za-z_$][\w$]*").match(self.t, self.i)
        if word and word.group(0) in ("true", "false", "null", "undefined"):
            self.i = word.end()
            return {"true": True, "false": False}.get(word.group(0))
        self.skip_expression()
        return _JS_UNKNOWN

    def selected(self):
        # [giá trị khi chọn, giá trị khi bỏ chọn]: chỉ lấy phần tử đầu, bỏ qua phần còn lại
        if self.peek() != "[":
            self.skip_expression()
            return _JS_UNKNOWN
        self.i += 1
        first = self.value()
        while self.peek() not in ("]", ""):
            if self.peek() == ",":
                self.i += 1
            else:
                self.skip_expression()
        self.i += 1
        return first

    def ident(self):
        m = _re.compile(r"[A-Za-z_$][\w$]*").match(self.t, self.i)
        if not m:
            raise ValueError(f"expected key at {self.i}")
        self.i = m.end()
        return m.group(0)


def _parse_toolkit_ui(text):
    archs = {}
    heads = list(_re.finditer(r'^    name: "([^"]+)",', text, _re.M))
    for n, head in enumerate(heads):
        end = heads[n + 1].start() if n + 1 < len(heads) else len(text)
        block = text[head.end():end]
        m = _re.search(r"^    defaults: \{", block, _re.M)
        if not m:
            continue
        r = _JsReader(block, m.end() - 1)
        r.i += 1
        defaults = {}
        while r.peek() not in ("}", ""):
            try:
                key = r.string()
                r.expect(":")
                val = r.selected()
            except (ValueError, IndexError):
                break
            if val is not _JS_UNKNOWN:
                defaults[key] = val
            if r.peek() == ",":
                r.i += 1
        archs[head.group(1)] = defaults
    return archs


_toolkit_defaults_cache = None

def toolkit_defaults(arch):
    """Thông số mặc định của ai-toolkit cho một arch, dạng {'model.qtype': 'qfloat8', ...}."""
    global _toolkit_defaults_cache
    if _toolkit_defaults_cache is None:
        _toolkit_defaults_cache = {}
        root = globals().get("toolkit_dir", "/content/SDVN-training-colab-flux/ai-toolkit")
        for f in sorted(_glob.glob(f"{root}/extensions_built_in/*/ui.tsx")):
            try:
                with open(f, encoding="utf-8") as fh:
                    _toolkit_defaults_cache.update(_parse_toolkit_ui(fh.read()))
            except Exception as e:
                print(f"⚠️ Không đọc được thông số mặc định từ {f}: {e}")
    out = {}
    for key, value in _toolkit_defaults_cache.get(arch, {}).items():
        if not key.startswith("config.process[0].") or value is None:
            continue
        path = key[len("config.process[0]."):].replace("datasets[x]", "datasets")
        if "." in path:  # bỏ khoá thay cả một mục lớn (VD: toàn bộ "sample")
            out[path] = value
    return out


def set_config_path(process, path, value):
    """Gán process[a][b]... = value theo đường dẫn 'a.b.c'; 'datasets.x' áp cho mọi dataset."""
    parts = path.split(".")
    nodes = process["datasets"] if parts[0] == "datasets" else [process]
    if parts[0] == "datasets":
        parts = parts[1:]
    for node in nodes:
        for p in parts[:-1]:
            if not isinstance(node.get(p), dict):
                node[p] = {}
            node = node[p]
        node[parts[-1]] = value

#Toolkit defaults

def config(folder):
  check_value(default_value)
  sample_prompt_ = Sampler_Prompt
  sample_image_path_ = Control_Img_Path
  with open(f'{config_dir}/config.yaml', 'r') as file:
    config_dict = yaml.safe_load(file)
  file_path = f'{config_dir}/config_{folder["name"]}.yaml'
  process = config_dict["config"]["process"][0]

  # Thông số ẩn (không có trên form Colab) lấy theo mặc định của ai-toolkit cho model đang train.
  # Các thông số có trên form được gán ở dưới nên vẫn theo lựa chọn của người dùng.
  arch_defaults = toolkit_defaults(type_config[TypeTrain][1])
  for key, value in arch_defaults.items():
    if key != "model.name_or_path":
      set_config_path(process, key, value)
  if not arch_defaults:
    print(f"⚠️ Không tìm thấy thông số mặc định của ai-toolkit cho {type_config[TypeTrain][1]}, dùng config.yaml")

  config_dict["config"]["name"] = Lora_name if len(folder_train) == 1 else folder["name"]

  config_dict["config"]["process"][0]["training_folder"] = OutputFolder
  config_dict["config"]["process"][0]["trigger_word"] = Custom_Caption if Custom_Caption != "" else folder["name"]

  config_dict["config"]["process"][0]["network"]["linear"] = Dim
  config_dict["config"]["process"][0]["network"]["linear_alpha"] = Alpha
  config_dict["config"]["process"][0]["save"]["save_every"] = Save_steps

  config_dict["config"]["process"][0]["datasets"][0]["folder_path"] = folder["path"]
  config_dict["config"]["process"][0]["datasets"][0]["control_path"] = folder.get("control_path", "") if ControlFolder != "" and  TypeTrain not in multi_control_types else None
  config_dict["config"]["process"][0]["datasets"][0]["control_path_1"] = folder.get("control_path", "") if ControlFolder != "" and  TypeTrain in multi_control_types else None
  config_dict["config"]["process"][0]["datasets"][0]["control_path_2"] = folder.get("control_path_2", "") if ControlFolder_2 != "" and  TypeTrain in multi_control_types else None
  config_dict["config"]["process"][0]["datasets"][0]["control_path_3"] = folder.get("control_path_3", "") if ControlFolder_3 != "" and  TypeTrain in multi_control_types else None
  config_dict["config"]["process"][0]["datasets"][0]["resolution"] = [int(x) for x in Resolution.split(',')]
  config_dict["config"]["process"][0]["datasets"][0]["cache_latents_to_disk"] = Cache_latents

  config_dict["config"]["process"][0]["train"]["batch_size"] = Batch_size
  config_dict["config"]["process"][0]["train"]["steps"] = folder["steps"]
  config_dict["config"]["process"][0]["train"]["train_text_encoder"] = Train_TexEncoder
  config_dict["config"]["process"][0]["train"]["lr"] = Lr
  config_dict["config"]["process"][0]["train"]["lr_scheduler"] = Lr_Scheduler
  config_dict["config"]["process"][0]["train"]["disable_sampling"] = False if Sampler_Steps > 0 else True
  config_dict["config"]["process"][0]["train"]["timestep_type"] = Timestep_type
  
  config_dict["config"]["process"][0]["model"]["name_or_path"] = type_config[TypeTrain][0]
  config_dict["config"]["process"][0]["model"]["arch"] = type_config[TypeTrain][1]
  config_dict["config"]["process"][0]["model"]["low_vram"] = Low_VRAM
  if is_g4 and G4_CONVROT8:
    for key in ("qtype", "qtype_te"):
      if process["model"].get(key) == "qfloat8":
        process["model"][key] = "convrot8"
  for key, value in type_overrides.get(TypeTrain, {}).items():
    set_config_path(process, key, value)
  print(f"⚙️ {TypeTrain}: quantize={process['model'].get('qtype')}/{process['model'].get('qtype_te')}, "
        f"guidance={process['sample'].get('guidance_scale')}, sample_steps={process['sample'].get('sample_steps')}")

  if Wandb_API != "":
    config_dict["config"]["process"][0]["logging"]["use_wandb"] = True
    config_dict["config"]["process"][0]["logging"]["run_name"] = f"{Lora_name}_{type_config[TypeTrain][1]}"
    os.environ['WANDB_API_KEY'] = Wandb_API
  trigger = Custom_Caption if Custom_Caption != "" else folder["name"]
  if folder.get("control_path", "") != "":
      s, p = random_sample(folder["path"],folder["control_path"], trigger)
  else:
      s, p = random_sample(folder["path"],folder["path"], trigger)
  if sample_prompt_ == "" :
      sample_prompt_ = s
  if sample_image_path_ == "":
      sample_image_path_ = p
  if TypeTrain in ctrl_sample_types and ControlFolder != "":
    sample_prompt_ = f"{sample_prompt_} --ctrl_img {sample_image_path_}"
    if sample_image_path_ != "":
      w, h = image_size(sample_image_path_)
      min_side = sample_min_side.get(TypeTrain, 0)
      if min(w, h) < min_side:
        scale = min_side / min(w, h)
        w, h = round(w * scale / 16) * 16, round(h * scale / 16) * 16
      process["sample"]["width"], process["sample"]["height"] = w, h
  print(sample_prompt_)
  config_dict["config"]["process"][0]["sample"]["sample_every"] = Sampler_Steps if Sampler_Steps > 0 else 100
  config_dict["config"]["process"][0]["sample"]["samples"][0]["prompt"] = sample_prompt_

  config_dict["meta"]["name"] = Lora_name if len(folder_train) == 1 else folder["name"]
  config_dict["meta"]["version"] = "Train by trainlora.vn | stablediffusion.vn"
  with open(file_path, 'w') as file:
      yaml.dump(config_dict, file, default_flow_style=False)

  print('=====================')
  print(f'Thư mục train: {folder["path"]}')
  print(f'  Số file train: {len(get_train_files(folder["path"]))}')
  print(f'  Số Steps: {folder["steps"]}')
  print(f'  Output: {OutputFolder}/{folder["name"]}')
  print('=====================')

def image_size(image_path, max_size=1536):
    with Image.open(image_path) as img:
        width, height = img.size
    if max(width, height) <= max_size:
        return width, height
    scale_ratio = max_size / max(width, height)
    new_width = int(width * scale_ratio)
    new_height = int(height * scale_ratio)
    return new_width, new_height

def hug_down(link,path):
  name = path.split('/')[-1]
  folder = path.split(name)[0]
  if "blob" in link:
    link = link.replace("blob","resolve")
  !aria2c --console-log-level=error --summary-interval=10 -c -x 8 -s 8 -k 1M {link} -d {folder} -o {name}

def train():
  %cd {toolkit_dir}
  delete_specific_models()
  for folder in folder_train:
    !python run.py {config_dir}/config_{folder["name"]}.yaml

def cloudflare_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('172.28.0.12', port))
      if result == 0:
        break
      sock.close()
  p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://172.28.0.12:{}".format(port)], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
  for line in p.stderr:
    l = line.decode()
    if "trycloudflare.com " in l:
      print(f"\033[92m{'🔗 Link online để sử dụng (delay 1 phút):'}\033[0m", l[l.find("http"):], end='')

def sever_flare(port):
  threading.Thread(target=cloudflare_thread, daemon=True, args=(port,)).start()

clear_output()